In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field

class Education(BaseModel):
    level: Optional[str] = Field(description="The degree or qualification (e.g., B.Tech, 12th Grade).")
    institution: Optional[str] = Field(description="The name of the educational institution.")
    board: Optional[str] = Field(description="The name of the board or university.")
    year: Optional[str] = Field(description="The year of completion or graduation.")
    percentage: Optional[str] = Field(description="The percentage or CGPA obtained.")

class WorkExperience(BaseModel):
    position: Optional[str] = Field(description="The job title or position held.")
    company: Optional[str] = Field(description="The name of the company.")
    duration: Optional[str] = Field(description="The start and end dates of the employment.")
    description: Optional[str] = Field(description="A description of responsibilities and achievements.")

class Suggestions(BaseModel):
    commonSkills: List[str] = Field(description="Widely used industry skills found in the resume (e.g., Python, SQL).")
    uniqueSkills: List[str] = Field(description="Specialized or niche skills found in the resume (e.g., LangChain, Huggingface).")


class ExtractedData(BaseModel):
    name: str = Field(description="The full name of the candidate.")
    email: Optional[str] = Field(description="The email address of the candidate.")
    phone: Optional[str] = Field(description="The phone number of the candidate.")
    skills: List[str] = Field(description="A list of all technical and soft skills mentioned.")
    cgpa: Optional[str] = Field(description="The final cumulative grade point average, if mentioned.")
    shortlisted: bool = Field(default=False, description="Set to false by default.")
    education: List[Education] = Field(description="A list of the candidate's educational qualifications.")
    workExperience: List[WorkExperience] = Field(description="A list of the candidate's previous work experiences.")
    suggestions: Suggestions = Field(description="Categorized list of skills based on their commonality.")

class ParsedResume(BaseModel):
    """The root model for the JSON output, containing all extracted resume data."""
    extractedData: ExtractedData = Field(description="Contains all the extracted information from the resume.")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./uploads/Curriculum_vitae.pdf" 
loader = PyPDFLoader(pdf_path)
documents = loader.load()
resume_text = "\n".join([doc.page_content for doc in documents])

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
parser = PydanticOutputParser(pydantic_object=ParsedResume)

prompt_template = """
You are an expert HR assistant specializing in parsing resume documents.
Your task is to accurately extract information from the provided resume text and format it into a JSON object.

Follow these instructions carefully:
1.  Extract the information based on the schema provided below.
2.  If a piece of information is not found in the resume, leave the corresponding field null.
3.  Be precise and do not invent information.

{format_instructions}

Here is the resume text:
---
{resume_text}
---
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["resume_text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm | parser

In [25]:
import json

try:
    parsed_result = chain.invoke({"resume_text": resume_text})

    json_output = parsed_result.model_dump_json(indent=2)

    print("done.")
    print("\n--- JSON Output ---")
    print(json_output)

except Exception as e:
    print(f"An error occurred: {e}")

parsed_data = json.loads(json_output)
with open("parsed_resume.json", "w") as f:
    json.dump(parsed_data, f, indent=2)

done.

--- JSON Output ---
{
  "extractedData": {
    "name": "AADITYA RAJ",
    "email": "helloaadityahere@gmail.com",
    "phone": "+91 7060225009",
    "skills": [
      "Python",
      "C/C++",
      "SQL",
      "Tensorflow",
      "PyTorch",
      "Keras",
      "Huggingface",
      "Langchain",
      "Git",
      "Docker",
      "Jupyter-Notebook",
      "VS Code",
      "PyCharm",
      "Pandas",
      "NumPy",
      "Matplotlib"
    ],
    "cgpa": null,
    "shortlisted": false,
    "education": [
      {
        "level": "Bachelor of Technology (Computer Science and Engineering)",
        "institution": "Birla Institute of Technology, Mesra",
        "board": null,
        "year": "Present",
        "percentage": "8.28"
      },
      {
        "level": "12th",
        "institution": "Delhi Public School, Ranchi",
        "board": null,
        "year": "2023",
        "percentage": "96.6%"
      }
    ],
    "workExperience": [
      {
        "position": "Research Assistant"